# Group 26 — School Supply Request Classification
**Task type:** Predictive classification
**System question:** Based on the recorded inputs, which category should the system assign for a school supply request: **Approve**, **Review**, or **Defer**?

| Item | Value |
|---|---|
| Primary algorithm | Random Forest Classifier |
| Comparison algorithm | Decision Tree Classifier |
| System inputs | Quantity, urgency, current stock, purpose |
| System output | Approve, Review, Defer |

Run the cells from top to bottom (**Runtime → Run all**). Part 5 launches the Streamlit app and prints a public link.

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 140)

## Part 1 — Understand the Dataset

**Reference-only fields (kept for records, not used by the model):** `request_id`, `requesting_office`, `date_requested`. They identify a request but do not explain whether it should be approved.

**Analytical inputs (features):**
- `quantity` — number of units requested (numeric)
- `urgency` — Low / Medium / High (categorical, ordered)
- `current_stock` — units currently available in the supply room (numeric)
- `purpose` — Instruction, Examination, School Event, Office Use, Personal (categorical)

**Target label (required for training):** `decision` — the category assigned by the supply officer in past records (Approve, Review, Defer).

**Generated output:** the model's predicted category plus its confidence (class probability).

**Missing information:** the dataset does not record the item's cost, the budget left for the term, supplier lead time, or who approved past requests. These would likely improve the decision but are outside our four required inputs.

**Concern (fairness):** if past officers favored some offices, the model would learn that bias. That is one reason `requesting_office` is excluded from the inputs. Also, "Personal" purpose requests get deferred often, so the system should always let a human override.

## Part 2 — Prepare and Check the Dataset

We simulate **90 historical requests** using rules that mimic a supply officer's judgment, then add a little noise (some officers decide differently) so the data is not perfectly predictable. A few deliberate problems (missing values, a duplicate, messy category spelling) are injected so we can demonstrate the cleaning checks.

In [ ]:
rng = np.random.default_rng(SEED)
N = 90

urgency_levels = ["Low", "Medium", "High"]
purposes = ["Instruction", "Examination", "School Event", "Office Use", "Personal"]
offices = ["Grade 7", "Grade 8", "Grade 9", "Grade 10", "Registrar", "Guidance", "Library", "Clinic"]

df = pd.DataFrame({
    "request_id": [f"REQ-{i:03d}" for i in range(1, N + 1)],
    "requesting_office": rng.choice(offices, N),
    "date_requested": pd.to_datetime("2026-06-01") + pd.to_timedelta(rng.integers(0, 120, N), unit="D"),
    "quantity": rng.integers(1, 151, N),
    "urgency": rng.choice(urgency_levels, N, p=[0.35, 0.40, 0.25]),
    "current_stock": rng.integers(0, 301, N),
    "purpose": rng.choice(purposes, N, p=[0.30, 0.20, 0.18, 0.20, 0.12]),
})

def officer_decision(r):
    """Simulated rule a supply officer follows."""
    enough_stock = r.current_stock >= r.quantity
    academic = r.purpose in ("Instruction", "Examination")
    if r.purpose == "Personal":
        return "Defer" if r.urgency != "High" else "Review"
    if not enough_stock:
        return "Review" if (r.urgency == "High" and academic) else "Defer"
    if r.quantity > 100:
        return "Review"
    if academic or r.urgency in ("Medium", "High"):
        return "Approve"
    return "Review"

df["decision"] = df.apply(officer_decision, axis=1)

# ~8% label noise: officers are not perfectly consistent
flip = rng.random(N) < 0.08
df.loc[flip, "decision"] = rng.choice(["Approve", "Review", "Defer"], flip.sum())

# Inject realistic data-entry problems to demonstrate cleaning
raw = df.copy()
raw.loc[[5, 33], "current_stock"] = np.nan
raw.loc[[12], "urgency"] = np.nan
raw.loc[[20], "urgency"] = "high "          # inconsistent spelling
raw.loc[[41], "purpose"] = "instruction"    # inconsistent case
raw = pd.concat([raw, raw.iloc[[7]]], ignore_index=True)   # duplicate row

raw.to_csv("school_supply_requests_raw.csv", index=False)
raw.head(10)

### Data dictionary

In [ ]:
data_dictionary = pd.DataFrame([
    ["request_id", "Reference only", "Text", "Unique request code", "REQ-001"],
    ["requesting_office", "Reference only", "Category", "Office or grade level that filed the request (excluded to avoid bias)", "Grade 9"],
    ["date_requested", "Reference only", "Date", "Date the request was filed", "2026-07-14"],
    ["quantity", "Input", "Integer (1–150)", "Units requested", "40"],
    ["urgency", "Input", "Category", "Low / Medium / High", "High"],
    ["current_stock", "Input", "Integer (0–300)", "Units currently in the supply room", "120"],
    ["purpose", "Input", "Category", "Instruction / Examination / School Event / Office Use / Personal", "Examination"],
    ["decision", "Target label", "Category", "Approve / Review / Defer (past officer decision)", "Approve"],
], columns=["Field", "Role", "Type", "Description", "Example"])
data_dictionary

### Quality checks: missing values, duplicates, invalid categories, target balance

In [ ]:
data = pd.read_csv("school_supply_requests_raw.csv")

print("Missing values per column:")
print(data.isna().sum(), "\n")
print("Duplicate rows:", data.duplicated().sum())
print("Invalid urgency values:", sorted(set(data.urgency.dropna()) - set(urgency_levels)))
print("Invalid purpose values:", sorted(set(data.purpose.dropna()) - set(purposes)))

In [ ]:
# Cleaning
clean = data.drop_duplicates().copy()

# Standardise category spelling
clean["urgency"] = clean["urgency"].str.strip().str.title()
clean["purpose"] = clean["purpose"].str.strip().str.title()

# Missing values: numeric -> median, categorical -> most frequent
clean["current_stock"] = clean["current_stock"].fillna(clean["current_stock"].median())
clean["urgency"] = clean["urgency"].fillna(clean["urgency"].mode()[0])

assert clean.isna().sum().sum() == 0
assert set(clean.urgency) <= set(urgency_levels) and set(clean.purpose) <= set(purposes)

clean["current_stock"] = clean["current_stock"].astype(int)
clean.to_csv("school_supply_requests_clean.csv", index=False)   # use this file in KNIME
print("Clean records:", len(clean))
clean.describe(include="all").T

In [ ]:
counts = clean["decision"].value_counts()
print(counts, "\n")
print((counts / counts.sum()).round(2))
counts.plot(kind="bar", color=["#4C72B0", "#DD8452", "#55A868"], title="Target balance")
plt.ylabel("Requests"); plt.xticks(rotation=0); plt.show()

## Part 3 — KNIME Workflow (reference)

Use `school_supply_requests_clean.csv` (download it from the Colab file panel on the left).

`CSV Reader → Column Filter → Missing Value → One to Many → Partitioning → [Random Forest Learner → Random Forest Predictor] + [Decision Tree Learner → Decision Tree Predictor] → Scorer (×2)`

| Node | Setting |
|---|---|
| CSV Reader | `school_supply_requests_clean.csv` |
| Column Filter | Exclude `request_id`, `requesting_office`, `date_requested` |
| Missing Value | Numbers → Median; Strings → Most frequent |
| One to Many | Columns `urgency`, `purpose` |
| Partitioning | Relative 75%, **Stratified sampling** on `decision`, seed 42 |
| Random Forest Learner | Target `decision`, 200 models, seed 42 |
| Decision Tree Learner | Class column `decision`, Gini index |
| Predictors | Connect learner model + Partitioning bottom port; tick "Append class probabilities" |
| Scorer | Actual `decision` vs `Prediction (decision)` → accuracy, confusion matrix; use "Accuracy statistics" port for precision, recall, F1 |

## Part 4 — Recreate the Process in Python

Same steps as KNIME: drop reference fields → one-hot encode `urgency` and `purpose` → 75/25 stratified split → train both classifiers → score. We also run 5-fold cross-validation because the dataset is small and one split can be lucky or unlucky.

In [ ]:
df = pd.read_csv("school_supply_requests_clean.csv")

FEATURES = ["quantity", "urgency", "current_stock", "purpose"]
NUMERIC = ["quantity", "current_stock"]
CATEGORICAL = ["urgency", "purpose"]
TARGET = "decision"
CLASSES = ["Approve", "Review", "Defer"]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)

def make_pipeline(model):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(categories=[urgency_levels, purposes], handle_unknown="ignore"), CATEGORICAL),
        ("num", "passthrough", NUMERIC),
    ])
    return Pipeline([("prep", pre), ("model", model)])

models = {
    "Random Forest (primary)": make_pipeline(
        RandomForestClassifier(n_estimators=200, max_depth=6, random_state=SEED)),
    "Decision Tree (comparison)": make_pipeline(
        DecisionTreeClassifier(max_depth=5, random_state=SEED)),
}
print("Train:", len(X_train), "| Test:", len(X_test))

In [ ]:
rows, preds = [], {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_test)
    preds[name] = p
    cvs = cross_validate(make_pipeline(pipe.named_steps["model"]), X, y, cv=cv,
                         scoring=["accuracy", "f1_macro"])
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, p),
        "Precision (macro)": precision_score(y_test, p, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, p, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_test, p, average="macro", zero_division=0),
        "CV Accuracy (5-fold)": cvs["test_accuracy"].mean(),
        "CV F1 (5-fold)": cvs["test_f1_macro"].mean(),
    })
    print(f"===== {name} =====")
    print(classification_report(y_test, p, labels=CLASSES, zero_division=0))

results = pd.DataFrame(rows).set_index("Model").round(3)
results

### Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, p) in zip(axes, preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, p, labels=CLASSES),
                           display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

### Comparison chart

In [ ]:
ax = results[["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)", "CV F1 (5-fold)"]].T.plot(
    kind="bar", figsize=(9, 4.5), color=["#4C72B0", "#DD8452"], rot=0)
ax.set_ylim(0, 1.05); ax.set_ylabel("Score"); ax.set_title("Random Forest vs Decision Tree")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

### Feature importance (Random Forest)

In [ ]:
rf = models["Random Forest (primary)"]
names = rf.named_steps["prep"].get_feature_names_out()
imp = pd.Series(rf.named_steps["model"].feature_importances_, index=names)
imp.index = imp.index.str.replace("cat__", "").str.replace("num__", "")
imp.sort_values().plot(kind="barh", figsize=(7, 4.5), color="#4C72B0", title="Random Forest feature importance")
plt.tight_layout(); plt.show()

### Save the selected pipeline

In [ ]:
best_name = results["CV F1 (5-fold)"].idxmax()
print("Selected model:", best_name)

final = make_pipeline(models[best_name].named_steps["model"])
final.fit(X, y)   # refit on all records before deployment
joblib.dump(final, "supply_request_model.joblib")
print("Saved -> supply_request_model.joblib")

## Part 5 — Streamlit Interface

The cell below writes `app.py`. It accepts a single request (form) **or** a CSV upload, validates the inputs, runs the saved pipeline, and shows the category with a short explanation and the model's confidence.

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
import streamlit as st

URGENCY = ["Low", "Medium", "High"]
PURPOSES = ["Instruction", "Examination", "School Event", "Office Use", "Personal"]
FEATURES = ["quantity", "urgency", "current_stock", "purpose"]
LOW_CONFIDENCE = 0.55

model = joblib.load("supply_request_model.joblib")

def validate(row):
    errors = []
    q, s = row.get("quantity"), row.get("current_stock")
    if pd.isna(q) or not (1 <= float(q) <= 500):
        errors.append("Quantity must be between 1 and 500.")
    if pd.isna(s) or not (0 <= float(s) <= 1000):
        errors.append("Current stock must be between 0 and 1000.")
    if str(row.get("urgency")).strip().title() not in URGENCY:
        errors.append("Urgency must be Low, Medium, or High.")
    if str(row.get("purpose")).strip().title() not in PURPOSES:
        errors.append("Purpose must be one of: " + ", ".join(PURPOSES) + ".")
    return errors

def explain(row, label):
    q, s = float(row["quantity"]), float(row["current_stock"])
    reasons = []
    reasons.append("stock covers the request" if s >= q else f"stock is short by {int(q - s)} units")
    reasons.append(f"urgency is {row['urgency']}")
    reasons.append(f"purpose is {row['purpose']}")
    if q > 100:
        reasons.append("the quantity is large")
    return f"**{label}** because " + ", ".join(reasons) + "."

def predict(row):
    row = dict(row)
    row["urgency"] = str(row["urgency"]).strip().title()
    row["purpose"] = str(row["purpose"]).strip().title()
    X = pd.DataFrame([row])[FEATURES]
    proba = model.predict_proba(X)[0]
    classes = model.classes_
    label = classes[proba.argmax()]
    return row, label, dict(zip(classes, proba))

st.set_page_config(page_title="Supply Request Classifier", page_icon="📦")
st.title("School Supply Request Classifier")
st.caption("Group 26 · Random Forest · outputs Approve, Review, or Defer")

tab1, tab2 = st.tabs(["Single request", "Upload CSV"])

with tab1:
    with st.form("req"):
        c1, c2 = st.columns(2)
        quantity = c1.number_input("Quantity requested", min_value=0, max_value=1000, value=20, step=1)
        stock = c2.number_input("Current stock", min_value=0, max_value=2000, value=100, step=1)
        urgency = c1.selectbox("Urgency", ["-- select --"] + URGENCY)
        purpose = c2.selectbox("Purpose", ["-- select --"] + PURPOSES)
        go = st.form_submit_button("Classify")
    if go:
        row = {"quantity": quantity, "urgency": urgency, "current_stock": stock, "purpose": purpose}
        errs = validate(row)
        if errs:
            for e in errs: st.error(e)
        else:
            row, label, probs = predict(row)
            conf = probs[label]
            color = {"Approve": "green", "Review": "orange", "Defer": "red"}[label]
            st.markdown(f"### Result: :{color}[{label}]")
            st.write(explain(row, label))
            st.write(f"Model confidence: **{conf:.0%}**")
            if conf < LOW_CONFIDENCE:
                st.warning("Low confidence — evidence is mixed. Send this request to the supply officer for manual review.")
            st.bar_chart(pd.Series(probs, name="probability"))

with tab2:
    st.write("CSV must contain columns: quantity, urgency, current_stock, purpose")
    file = st.file_uploader("Upload CSV", type="csv")
    if file:
        up = pd.read_csv(file)
        missing_cols = [c for c in FEATURES if c not in up.columns]
        if missing_cols:
            st.error(f"Missing columns: {missing_cols}")
        else:
            out = []
            for _, r in up.iterrows():
                errs = validate(r)
                if errs:
                    out.append({"prediction": "Invalid", "confidence": None, "note": " ".join(errs)})
                else:
                    rr, label, probs = predict(r[FEATURES].to_dict())
                    note = "Low confidence - manual review" if probs[label] < LOW_CONFIDENCE else explain(rr, label).replace("**", "")
                    out.append({"prediction": label, "confidence": round(probs[label], 2), "note": note})
            result = pd.concat([up.reset_index(drop=True), pd.DataFrame(out)], axis=1)
            st.dataframe(result)
            st.download_button("Download results", result.to_csv(index=False), "predictions.csv")

### Run the app inside Colab
Run the next cell. It prints a **tunnel password** (your Colab IP) and a `https://....loca.lt` link. Open the link, paste the password, and the app loads. Stop the cell to shut the app down.

In [ ]:
!pip -q install streamlit
!npm -q install localtunnel
import urllib.request
print("Tunnel password:", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
!streamlit run app.py --server.port 8501 &>/content/streamlit_log.txt & npx localtunnel --port 8501

## Part 6 — Test, Explain, and Reflect

### Test cases
We run the same validation and prediction logic the app uses on four kinds of cases.

In [ ]:
VALID_U, VALID_P = set(urgency_levels), set(purposes)

def check_and_predict(case):
    errs = []
    if pd.isna(case.get("quantity")) or not (1 <= case["quantity"] <= 500): errs.append("bad quantity")
    if pd.isna(case.get("current_stock")) or not (0 <= case["current_stock"] <= 1000): errs.append("bad stock")
    if case.get("urgency") not in VALID_U: errs.append("bad urgency")
    if case.get("purpose") not in VALID_P: errs.append("bad purpose")
    if errs:
        return "Rejected (input error)", None, ", ".join(errs)
    proba = final.predict_proba(pd.DataFrame([case])[FEATURES])[0]
    label = final.classes_[proba.argmax()]
    note = "Low confidence -> manual review" if proba.max() < 0.55 else ""
    return label, round(proba.max(), 2), note

tests = [
    ("Normal",    "Exam papers, stock plenty", dict(quantity=30,  urgency="High",   current_stock=200, purpose="Examination")),
    ("Normal",    "Personal, low urgency",     dict(quantity=10,  urgency="Low",    current_stock=150, purpose="Personal")),
    ("Normal",    "Event, stock short",        dict(quantity=120, urgency="Low",    current_stock=40,  purpose="School Event")),
    ("Missing",   "No urgency given",          dict(quantity=25,  urgency=None,     current_stock=80,  purpose="Instruction")),
    ("Missing",   "No stock value",            dict(quantity=25,  urgency="Medium", current_stock=np.nan, purpose="Office Use")),
    ("Boundary",  "Quantity = stock",          dict(quantity=100, urgency="Medium", current_stock=100, purpose="Instruction")),
    ("Boundary",  "Quantity just over 100",    dict(quantity=101, urgency="Medium", current_stock=300, purpose="Office Use")),
    ("Boundary",  "Zero stock",                dict(quantity=1,   urgency="High",   current_stock=0,   purpose="Examination")),
    ("Boundary",  "Invalid quantity 0",        dict(quantity=0,   urgency="Low",    current_stock=50,  purpose="Instruction")),
    ("Uncertain", "Personal but High urgency", dict(quantity=60,  urgency="High",   current_stock=55,  purpose="Personal")),
    ("Uncertain", "Office use, stock barely short", dict(quantity=52, urgency="High", current_stock=50, purpose="Office Use")),
]
out = []
for kind, desc, case in tests:
    label, conf, note = check_and_predict(case)
    out.append({"Type": kind, "Case": desc, **case, "Prediction": label, "Confidence": conf, "Note": note})
pd.DataFrame(out)

### Critical-thinking questions

**1. Which input features should have the strongest influence?** Current stock and quantity should matter most, because together they tell us whether the request can physically be filled. Purpose comes next since instructional and exam needs should be prioritized over personal ones, and urgency acts as a tie-breaker. The Random Forest importance chart above shows whether the model learned this same order.

**2. What should the system do when evidence is incomplete or contradictory?** It should not guess. Missing or invalid fields are rejected with a clear message, and when the model's top probability is low (below 55%), the app flags the request for manual review instead of presenting the label as final. Contradictory cases such as a high-urgency personal request are exactly the ones a human officer should look at.

**3. What could happen if the system is wrong?** A wrong Approve can drain stock that teachers need for classes or exams. A wrong Defer can delay materials for a lesson or event and frustrate staff. Repeated errors against one office could feel unfair. For this reason the system is a decision aid only, and the supply officer keeps the final say.

### Interpretation
Both models were trained on the same 75% split and tested on the same 25%. On the single 23-record test split the two models tied (about 78% accuracy and 0.78 macro F1), but 5-fold cross-validation, which tests every record once, shows a clearer gap: the Random Forest averaged about 81% accuracy and 0.80 F1 against about 74% and 0.70 for the Decision Tree. Because it averages many trees, the Random Forest is less sensitive to which records land in training, which is why it is the selected model. The single Decision Tree is easier to read as rules, but it overfits more easily. Most mistakes happen around the Review class, since Review sits between the other two categories and is the most "judgment-based" label. The boundary tests also show a weak spot: a request with zero stock was still predicted Approve because the dataset has very few tiny-quantity, zero-stock examples, so a simple rule check (stock < quantity) is worth adding before the model.

### Limitations
The dataset is small (90 clean records) and simulated from assumed officer rules, so the scores show how well the models learn those rules, not how they would perform in a real school. Around 8% of labels were randomly changed to imitate inconsistent decisions, which caps the achievable accuracy. Important factors such as item cost, budget, and supplier lead time are not included. A real deployment would need actual historical requests, periodic retraining, and a check that no office is treated unfairly.